In [ ]:
%run ./_utils

In [ ]:
from pyspark.sql import functions as F

#### Create `openalex.works.openalex_references_snapshot` in the shape `/works/{id}/references` serves (oxjob #2347)
One row per work with a reference list: `id`, `references` (each `{raw_reference_string, raw_doi, work_id}`, full URLs) and
`updated_date`, when the list last changed (the Lakebase copy's `updated_at`), so daily syncs fetch only changed lists.
Reads what the References On Works job built the afternoon before, so lists run one day behind the works.

In [ ]:
changed = (spark.read.table("openalex.works.lakebase_work_references_0").select("work_id", "updated_at")
           .unionByName(spark.read.table("openalex.works.lakebase_work_references_1").select("work_id", "updated_at")))
df = (spark.read.table("openalex.works.references_on_works")
      .join(changed, "work_id", "left")
      .select(
          F.concat(F.lit("https://openalex.org/W"), F.col("work_id")).alias("id"),
          F.transform("references", lambda x: F.struct(
              x["raw_reference_string"].alias("raw_reference_string"),
              x["raw_doi"].alias("raw_doi"),
              F.when(x["work_id"].isNotNull(), F.concat(F.lit("https://openalex.org/W"), x["work_id"])).alias("work_id"),
          )).alias("references"),
          F.coalesce("updated_at", "built_at").alias("updated_date"),
      ))
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("openalex.works.openalex_references_snapshot")

#### Export to JSONL + Parquet on S3
Writes both formats under `s3://openalex-snapshots/full/{date}/{format}/references/`.

In [ ]:
date_str = get_snapshot_date()
df = spark.read.table("openalex.works.openalex_references_snapshot")
export_partitioned_all_formats(spark, dbutils, df, date_str, "references", salt=True)